In [0]:
from pyspark.sql.functions import *

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.dim_tiempo (
    fecha DATE COMMENT 'Fecha en formato yyyy-MM-dd',
    anio INT COMMENT 'Año derivado de la fecha',
    mes STRING COMMENT 'Mes derivado de la fecha',
    dia INT COMMENT 'Día derivado de la fecha',
    estacion STRING COMMENT 'Estación del año según mes',
    fecha_data DATE COMMENT 'Fecha de actualización de la tabla'
) USING DELTA;

In [0]:
dim_tiempo = (
    spark.table("silver.hd_lima_air_pollution")
    .select("fecha","anio","mes","dia").dropDuplicates()
    .withColumn("estacion",
        expr("""
            CASE
                WHEN fecha >= to_date(concat(year(fecha), '-12-21')) OR fecha < to_date(concat(year(fecha), '-03-21')) THEN 'Verano'
                WHEN fecha >= to_date(concat(year(fecha), '-03-21')) AND fecha < to_date(concat(year(fecha), '-06-21')) THEN 'Otoño'
                WHEN fecha >= to_date(concat(year(fecha), '-06-21')) AND fecha < to_date(concat(year(fecha), '-09-23')) THEN 'Invierno'
                WHEN fecha >= to_date(concat(year(fecha), '-09-23')) AND fecha < to_date(concat(year(fecha), '-12-21')) THEN 'Primavera'
                ELSE 'Desconocido'
            END
        """)
    )
    .withColumn("mes",
        expr("""
            CASE
                WHEN mes == 1 THEN 'Enero'
                WHEN mes == 2 THEN 'Febrero'
                WHEN mes == 3 THEN 'Marzo'
                WHEN mes == 4 THEN 'Abril'
                WHEN mes == 5 THEN 'Mayo'
                WHEN mes == 6 THEN 'Junio'
                WHEN mes == 7 THEN 'Julio'
                WHEN mes == 8 THEN 'Agosto'
                WHEN mes == 9 THEN 'Septiembre'
                WHEN mes == 10 THEN 'Octubre'
                WHEN mes == 11 THEN 'Noviembre'
                WHEN mes == 12 THEN 'Diciembre'
                ELSE 'Desconocido'
            END
        """)
    )
    .withColumn("fecha_data", current_date())
)

dim_tiempo.write.format("delta").mode("overwrite").saveAsTable("gold.dim_tiempo")